Use autograd to find the gradient vector of f(x, y) = sin(x2 y) at the point (x, y) = (1.2, 3.4).

In [1]:
import torch

def f(X):
    return torch.sin(X[0]**2 * X[1])

X = torch.tensor([1.2, 3.4], requires_grad=True)
z = f(X)
z.backward()

print(X.grad)

tensor([1.4899, 0.2629])


Create a custom Dense module that replicates the functionality of an nn.Linear module followed by an nn.ReLU module. Try implementing it first using the nn.Linear and nn.ReLU modules, and then reimplement it using nn.Parameter and the relu() function.

In [2]:
import torch.nn as nn

class DenseCustom(nn.Module):
    def __init__(self, n_inputs, n_outputs):
        super().__init__()
        self.dense = nn.Sequential(
            nn.Linear(n_inputs, n_outputs), 
            nn.ReLU()
        )

    def forward(self, X):
        return self.dense(X)


In [3]:
class DenseCustomWithParam(nn.Module):
    def __init__(self, n_inputs, n_outputs):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n_outputs, n_inputs))
        self.b = nn.Parameter(torch.randn(n_outputs, ))
        self.relu = nn.ReLU()

    def forward(self, X):
        return self.relu(X @ self.W.T + self.b)

Build and train a classification MLP on the CoverType dataset.

In [4]:
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# Loading dataset
DATA_HOME = "../datasets"
covtype = fetch_covtype(data_home=DATA_HOME)
X, y = covtype.data, covtype.target

# Targets in covtype range from 1 to 7, Pytorch requirs index from 0
y = y - 1

# Split train/test
X_trainfull, X_test, y_trainfull, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardization 
scaler = StandardScaler()
scaler.fit_transform(X_trainfull)
scaler.transform(X_test)

# Split train/valid
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainfull, y_trainfull, test_size=0.2, random_state=42, stratify=y_trainfull
)

# Custom Datasets
class CovtypeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = CovtypeDataset(X_train, y_train)
valid_dataset = CovtypeDataset(X_valid, y_valid)
test_dataset = CovtypeDataset(X_test, y_test)

In [5]:
# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [6]:
class MLPClassifier(nn.Module):
    def __init__(self, n_inputs, n_classes, n_hidden, n_layers):
        super().__init__()
        layers = []
        in_features = n_inputs
        for _ in range(n_layers - 1):
            layers.append(nn.Linear(in_features, n_hidden))
            layers.append(nn.ReLU())
            in_features = n_hidden
        layers.append(nn.Linear(in_features, n_classes))
        self.model = nn.Sequential(*layers)

    def forward(self, X):
        return self.model(X)


# Full training function (train and eval)
def train(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs=15, device="cpu"):
    model.train()
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate(model, valid_loader, metric, device).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")

    return history



# Training function per epoch for pruning
def train_one_epoch(model, optimizer, criterion, data_loader, device="cpu"):
    model.train()
    total_loss = 0
    for X_batch, y_batch in data_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
    return total_loss / len(data_loader)

# Eval function
def evaluate(model, data_loader, metric, device="cpu"):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute().item()

In [ ]:
import optuna
import torchmetrics

def objective(trial):
    # Hyperparamètres à optimiser
    n_hidden = trial.suggest_int("n_hidden", 20, 300, log=True)
    n_layers = trial.suggest_int("n_layers", 2, 5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    #batch_size = trial.suggest_categorical("batch_size", [32, 64])

    # DataLoaders with desired batch size
    #train_loader_trial = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    #valid_loader_trial = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

    n_inputs = X_train.shape[1]
    n_classes = len(set(y_train))

    model = MLPClassifier(n_inputs, n_classes, n_hidden, n_layers)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    metric = torchmetrics.Accuracy(task="multiclass", num_classes=n_classes).to(device)
    
    train(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs=10, device=device)
    accuracy = evaluate(model, valid_loader, metric, device)

    return accuracy


def objective_pruning(trial, n_epochs = 15):
    # Hyperparamètres à optimiser
    n_hidden = trial.suggest_int("n_hidden", 20, 300, log=True)
    n_layers = trial.suggest_int("n_layers", 2, 5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    # DataLoaders with desired batch size
    train_loader_trial = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_loader_trial = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

    n_inputs = X_train.shape[1]
    n_classes = len(set(y_train))

    model = MLPClassifier(n_inputs, n_classes, n_hidden, n_layers).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    metric = torchmetrics.Accuracy(task="multiclass", num_classes=n_classes).to(device)

    best_validation_accuracy = 0.0

    for epoch in range(n_epochs):
        train_one_epoch(model, optimizer, criterion, train_loader, device=device)
        validation_accuracy = evaluate(model, valid_loader, metric, device)
        best_validation_accuracy = max(best_validation_accuracy, validation_accuracy)

        trial.report(validation_accuracy, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return best_validation_accuracy

In [9]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3, interval_steps=1)
study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
study.optimize(objective_pruning, n_trials=30)

print("Best hyperparameters :", study.best_params)
print("Best validation accuracy  :", study.best_value)

[I 2026-08-31 14:27:37,119] A new study created in memory with name: no-name-bc271826-a200-4cfb-aee3-635ed8ec1d95
[I 2026-08-31 14:32:33,862] Trial 0 finished with value: 0.7340525984764099 and parameters: {'n_hidden': 54, 'n_layers': 5, 'learning_rate': 0.0029106359131330704, 'batch_size': 32}. Best is trial 0 with value: 0.7340525984764099.
[I 2026-08-31 14:37:44,700] Trial 1 finished with value: 0.7573955059051514 and parameters: {'n_hidden': 23, 'n_layers': 5, 'learning_rate': 0.0015930522616241021, 'batch_size': 128}. Best is trial 1 with value: 0.7573955059051514.
[I 2026-08-31 14:41:16,935] Trial 2 finished with value: 0.7607302069664001 and parameters: {'n_hidden': 190, 'n_layers': 2, 'learning_rate': 0.0002310201887845295, 'batch_size': 128}. Best is trial 2 with value: 0.7607302069664001.
[I 2026-08-31 14:45:20,144] Trial 3 finished with value: 0.7370753884315491 and parameters: {'n_hidden': 64, 'n_layers': 3, 'learning_rate': 0.0016738085788752138, 'batch_size': 128}. Best i

Meilleurs hyperparamètres : {'n_hidden': 222, 'n_layers': 5, 'learning_rate': 0.00019469256284814212, 'batch_size': 128}
Meilleure accuracy validation : 0.8883522152900696


In [12]:
import pandas as pd

# Print top 5 trials 
df = study.trials_dataframe()
df_completed = df[df["state"] == "COMPLETE"]
top5 = df_completed.sort_values("value", ascending=False).head(5)
colonnes = ["number", "value"] + [col for col in df.columns if col.startswith("params_")]
top5[colonnes]

,number,value,params_batch_size,params_learning_rate,params_n_hidden,params_n_layers
23,23,0.888352,128,0.000195,222,5
28,28,0.872539,128,0.000310,300,5
24,24,0.868086,128,0.000339,240,5
25,25,0.867258,128,0.000372,236,5
26,26,0.859104,128,0.000433,236,5


In [25]:
from torch.utils.data import ConcatDataset

n_inputs = X_train.shape[1]
n_classes = len(set(y_train))

best_params = {
    "n_layers": 5,
    "n_hidden": 256,
    "learning_rate": 2.5e-4,
    "batch_size":128,
}

full_train_dataset = ConcatDataset([train_dataset, valid_dataset])
full_train_loader = DataLoader(full_train_dataset,
                                 batch_size=best_params["batch_size"],
                                 shuffle=True)


final_model = MLPClassifier(
    n_inputs, n_classes,
    n_hidden=best_params["n_hidden"],
    n_layers=best_params["n_layers"]
).to(device)

# Learning rate planning with warmup and decay 
training_params = [[best_params["learning_rate"]/3, 5], [best_params["learning_rate"], 20], [best_params["learning_rate"]/3, 5]]

for learning_rate, n_epochs in training_params:
    optimizer = torch.optim.Adam(final_model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    for _ in range(n_epochs):
        train_one_epoch(final_model, optimizer, criterion, full_train_loader, device)

In [26]:
metric= torchmetrics.Accuracy(task="multiclass", num_classes=n_classes).to(device)
test_accuracy = evaluate(final_model, test_loader, metric, device)
print("Final model accuracy on test set : ", test_accuracy)

Final model accuracy on test set :  0.9384525418281555
